# AIC 2026 - Object descriptions on Kaggle

Thin launcher only. Enable a GPU and attach private competition data. Internet-off also requires the private source snapshot and checksummed runtime mirror documented in `docs/cloud-runbook.md`.

In [ ]:
from pathlib import Path
import os, shutil, subprocess

source = Path("/kaggle/input/aic2026-source/AIC-2026")
target = Path("/kaggle/working/AIC-2026")
if not target.is_dir():
    if source.is_dir():
        shutil.copytree(source, target)
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("AIC_GITHUB_TOKEN")
        clone_env = os.environ.copy()
        clone_env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader", "GIT_CONFIG_VALUE_0": f"Authorization: Bearer {token}"})
        subprocess.run(["git", "clone", "https://github.com/AIVIETNAM-AIO-Dewey/AIC-2026.git", str(target)], env=clone_env, check=True)
        del token, clone_env
os.chdir(target)
print(Path.cwd())

Replace the attached Dataset slug and video ID. The cell detects an offline runtime mirror when attached; `/kaggle/input` remains read-only.

In [ ]:
from pathlib import Path
import os

os.environ["AIC_DATA_ROOT"] = "/kaggle/input/aic2026-data"
os.environ["AIC_ARTIFACT_ROOT"] = "/kaggle/working/aic2026-artifacts"
os.environ["AIC_VIDEO_ID"] = "replace_with_video_id"
mirror = Path("/kaggle/input/aic2026-runtime")
if mirror.is_dir():
    os.environ.update({"AIC_CACHE_ROOT": str(mirror / "cache"), "AIC_WHEELHOUSE": str(mirror / "wheelhouse"), "AIC_DAM_WHEEL": str(mirror / "wheelhouse/dam-1.0.0-py3-none-any.whl"), "AIC_DAM_CODE_REVISION": "153ad3d33c29324e9197f565547c6bc8500da02d", "HF_HUB_OFFLINE": "1"})
else:
    os.environ["AIC_CACHE_ROOT"] = "/kaggle/working/aic2026-model-cache"

In [ ]:
%%bash
set -euo pipefail
if [[ "${HF_HUB_OFFLINE:-0}" == "1" ]]; then
  cd /kaggle/input/aic2026-runtime
  sha256sum -c SHA256SUMS
  cd /kaggle/input/aic2026-source
  sha256sum -c SHA256SUMS
  cd /kaggle/working/AIC-2026
  python -m pip install --no-index --no-deps --find-links "$AIC_WHEELHOUSE" -r requirements/kaggle-offline.txt
  python -m pip install --no-index --no-deps "$AIC_DAM_WHEEL"
else
  python -m pip install --no-deps -r requirements/kaggle.txt
fi
python scripts/verify_environment.py \
  --config configs/offline/object_description.yaml \
  --device cuda --write-report

Smoke run on two frames uses an isolated `smoke/` artifact tree. For a full run, use a different output tree and remove `--limit 2`; never resume a limited artifact as a full run.

In [ ]:
%%bash
set -euo pipefail
SMOKE_ROOT="$AIC_ARTIFACT_ROOT/smoke"
FRAME_MANIFEST="$SMOKE_ROOT/frame_manifests/$AIC_VIDEO_ID.jsonl"
MASK_ARTIFACT="$SMOKE_ROOT/object_description/masks/$AIC_VIDEO_ID.jsonl"
DESCRIPTION_ARTIFACT="$SMOKE_ROOT/object_description/descriptions/$AIC_VIDEO_ID.jsonl"
MAP_CSV="$AIC_DATA_ROOT/map-keyframes/$AIC_VIDEO_ID.csv"
FRAMES_DIR="$AIC_DATA_ROOT/keyframes/$AIC_VIDEO_ID"
OBJECTS_DIR="$AIC_DATA_ROOT/objects/$AIC_VIDEO_ID"
python scripts/build_frame_manifest.py --config configs/offline/object_description.yaml --video-id "$AIC_VIDEO_ID" --map-csv "$MAP_CSV" --frames-dir "$FRAMES_DIR" --output "$FRAME_MANIFEST" --resume --limit 2
python scripts/prepare_object_masks.py --config configs/offline/object_description.yaml --video-id "$AIC_VIDEO_ID" --frame-manifest "$FRAME_MANIFEST" --objects-dir "$OBJECTS_DIR" --output "$MASK_ARTIFACT" --device cuda --resume --limit 2
python scripts/run_dam_descriptions.py --config configs/offline/object_description.yaml --video-id "$AIC_VIDEO_ID" --mask-artifact "$MASK_ARTIFACT" --output "$DESCRIPTION_ARTIFACT" --device cuda --resume --limit 2
python scripts/validate_object_artifacts.py --artifact "$DESCRIPTION_ARTIFACT" --manifest "${DESCRIPTION_ARTIFACT%.jsonl}.manifest.json" --require-captions